# Homework 4: Data (E)TL — Purchase-to-Pay Process
## Overview
This notebook transforms raw data extracted from multiple operational systems into a unified Activity Instance Log for the Purchase-to-Pay (P2P) process, ready for process mining analysis in Apromore.

**Six source tables:**
| Table | Description |
|---|---|
| `pr_info_table.csv` | Master data for Purchase Requisitions (case attributes) |
| `pr_po_table.csv` | Mapping between Purchase Requisitions (PR) and Purchase Orders (PO) |
| `purchasing_table.csv` | Purchasing activities (start+end timestamps, PR-level) |
| `requisition_states_table.csv` | State transitions for PRs/POs (instantaneous events) |
| `logistics_table.csv` | Logistics activities (paired start/end events, PO-level) |
| `finance_table.csv` | Finance activities (start+end timestamps, PO-level) |


## Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)

# Load all tables
finance    = pd.read_csv('finance_table.csv')
logistics  = pd.read_csv('logistics_table.csv')
pr_info    = pd.read_csv('pr_info_table.csv')
pr_po      = pd.read_csv('pr_po_table.csv')
purchasing = pd.read_csv('purchasing_table.csv')
req_states = pd.read_csv('requisition_states_table.csv')

print("All tables loaded successfully.")


All tables loaded successfully.


---
## Task 1: Data Inspection

Before transforming any data, we inspect each table to understand its structure, content and semantics.


### 1.1 `pr_info_table.csv` — Purchase Requisition Master Data

In [2]:
print("Shape:", pr_info.shape)
print()
pr_info.head(5)


Shape: (2500, 6)



,PRID,InitialRequisitionValue,PurchasingCategory,VendorID,Urgency,CostCenter
0,PR-00000,2845.05,Office Supplies,vendor-00162,False,CC-04
1,PR-00001,4686.84,Raw Materials,vendor-00171,False,CC-07
2,PR-00002,2356.78,Mechanical Components,vendor-00102,True,CC-05
3,PR-00003,3010.28,Office Supplies,vendor-00121,False,CC-01
4,PR-00004,3061.73,Subscriptions,vendor-00111,False,CC-02


In [3]:
print("Dtypes:")
print(pr_info.dtypes)
print()
print("Unique values / distributions:")
print("  PurchasingCategory:", pr_info['PurchasingCategory'].nunique(), "unique values")
print(pr_info['PurchasingCategory'].value_counts().head())
print()
print("  Urgency distribution:")
print(pr_info['Urgency'].value_counts())
print()
print("  CostCenter distribution:")
print(pr_info['CostCenter'].value_counts())
print()
print("  InitialRequisitionValue stats:")
print(pr_info['InitialRequisitionValue'].describe())


Dtypes:
PRID                           str
InitialRequisitionValue    float64
PurchasingCategory             str
VendorID                       str
Urgency                       bool
CostCenter                     str
dtype: object

Unique values / distributions:
  PurchasingCategory: 7 unique values
PurchasingCategory
Office Supplies          615
Raw Materials            433
Mechanical Components    414
IT Services              363
Marketing                338
Name: count, dtype: int64

  Urgency distribution:
Urgency
False    1929
True      571
Name: count, dtype: int64

  CostCenter distribution:
CostCenter
CC-02    398
CC-04    369
CC-05    358
CC-01    352
CC-03    351
CC-07    337
CC-06    332
CC-08      3
Name: count, dtype: int64

  InitialRequisitionValue stats:
count    2500.000000
mean     3520.820200
std      1102.384076
min       167.660000
25%      2762.467500
50%      3500.965000
75%      4235.082500
max      7921.320000
Name: InitialRequisitionValue, dtype: float64


**Description:** This is a **case attribute table** — one row per Purchase Requisition (PR). It stores static metadata that characterises each procurement case:
- `PRID`: Unique case identifier (e.g. `PR-00000`). This is the primary key.
- `InitialRequisitionValue`: The monetary value originally requested.
- `PurchasingCategory`: Type of goods/services being procured (e.g. Raw Materials, Office Supplies).
- `VendorID`: The supplier selected for this requisition.
- `Urgency`: Boolean flag indicating whether the request is urgent.
- `CostCenter`: The organisational unit bearing the cost.

This table contains **no event data** — it will be joined as case attributes in the final log.


### 1.2 `pr_po_table.csv` — PR–PO Mapping

In [4]:
print("Shape:", pr_po.shape)
print()
pr_po.head(5)


Shape: (1830, 2)



,PR,PO
0,PR-01179,PO-0001
1,PR-00335,PO-0002
2,PR-01065,PO-0006
3,PR-01982,PO-0008
4,PR-00940,PO-0011


In [5]:
print("PRs per PO (should be 1-to-1):")
print(pr_po.groupby('PO')['PR'].count().describe())
print()
print("POs per PR (should be 1-to-1):")
print(pr_po.groupby('PR')['PO'].count().describe())
print()
print("PRs in pr_info but NOT in pr_po (no PO created):", 
      pr_info[~pr_info['PRID'].isin(pr_po['PR'])]['PRID'].nunique())


PRs per PO (should be 1-to-1):
count    1830.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: PR, dtype: float64

POs per PR (should be 1-to-1):
count    1830.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: PO, dtype: float64

PRs in pr_info but NOT in pr_po (no PO created): 670


**Description:** This is a **relationship/bridge table** linking Purchase Requisitions to Purchase Orders. It has exactly one row per (PR, PO) pair, and the relationship is strictly 1-to-1: each PR maps to at most one PO and vice versa.

Importantly, 670 out of 2,500 PRs have no corresponding PO — these correspond to requisitions that were rejected or otherwise never progressed to a purchase order stage.

This table is essential for joining PO-level event tables (logistics, finance) back to the PR case identifier.


### 1.3 `purchasing_table.csv` — Purchasing Activities

In [6]:
print("Shape:", purchasing.shape)
print()
purchasing.head(5)


Shape: (10394, 7)



,purchase_requisition,employee,label,cost,requisition_value,timestamp_1,timestamp_2
0,PR-00000,Operations Requester-000001,Create Purchase Requisition,69.59,2845.05,2026-04-22T07:00:00.000000+0000,2026-04-22T08:09:35.186000+0000
1,PR-00001,Operations Requester-000002,Create Purchase Requisition,158.30,4686.84,2026-04-22T07:05:28.919000+0000,2026-04-22T09:43:47.141000+0000
2,PR-00000,Purchasing Officer-000001,Analyze Purchase Requisition,35.10,2845.05,2026-04-22T08:09:35.186000+0000,2026-04-22T08:51:42.050000+0000
3,PR-00002,Operations Requester-000003,Create Purchase Requisition,111.18,2356.78,2026-04-22T08:36:47.985000+0000,2026-04-22T10:27:58.722000+0000
4,PR-00000,Purchasing Officer-000002,Create Purchase Order,1.35,2845.05,2026-04-22T08:51:44.050000+0000,2026-04-22T08:53:21.313000+0000


In [7]:
print("Activity labels:")
print(purchasing['label'].value_counts())
print()
print("Sample case trace (PR-00001):")
print(purchasing[purchasing['purchase_requisition']=='PR-00001']
      .sort_values('timestamp_1')[['purchase_requisition','label','employee','timestamp_1','timestamp_2']].to_string())


Activity labels:
label
Analyze Purchase Requisition    3151
Create Purchase Requisition     2500
Create Purchase Order           2046
Confirm PO with Vendor          2046
Amend Purchase Requisition       651
Name: count, dtype: int64

Sample case trace (PR-00001):
   purchase_requisition                         label                     employee                      timestamp_1                      timestamp_2
1              PR-00001   Create Purchase Requisition  Operations Requester-000002  2026-04-22T07:05:28.919000+0000  2026-04-22T09:43:47.141000+0000
8              PR-00001  Analyze Purchase Requisition    Purchasing Officer-000005  2026-04-22T09:43:47.141000+0000  2026-04-22T09:46:12.454000+0000
24             PR-00001    Amend Purchase Requisition  Operations Requester-000004  2026-04-22T13:27:11.698000+0000  2026-04-23T07:57:11.318000+0000
30             PR-00001  Analyze Purchase Requisition    Purchasing Officer-000018  2026-04-23T07:57:11.318000+0000  2026-04-23T09:06:25.87

**Description:** This table records **purchasing department activities** at the PR level. Each row is an activity instance with:
- `purchase_requisition`: Case identifier (PR ID).
- `employee`: The resource who performed the activity.
- `label`: Activity name — one of: *Create Purchase Requisition, Analyze Purchase Requisition, Amend Purchase Requisition, Create Purchase Order, Confirm PO with Vendor*.
- `timestamp_1` / `timestamp_2`: Start and end time of the activity. The table is already in activity-instance format (start + end per row).
- `cost`: The labor/processing cost of this activity.
- `requisition_value`: The monetary value of the requisition at the time of this activity.

The trace for PR-00001 reveals a rework pattern: *Analyze* is followed by *Amend*, then *Analyze* again before approval — indicating a review-and-rework loop.


### 1.4 `requisition_states_table.csv` — State Transitions

In [8]:
print("Shape:", req_states.shape)
print()
req_states.head(5)


Shape: (6376, 3)



,RequisitionID,State,Timestamp
0,PR-00000,Purchase Requisition Approved,2026-04-22T08:51:43.050000+0000
1,PR-00002,Purchase Requisition Approved,2026-04-22T10:35:08.574000+0000
2,PR-00003,Purchase Requisition Approved,2026-04-22T10:37:55.319000+0000
3,PR-00005,Purchase Requisition Approved,2026-04-22T12:17:48.311000+0000
4,PR-00006,Purchase Requisition Approved,2026-04-23T08:32:16.187000+0000


In [9]:
print("State values:")
print(req_states['State'].value_counts())
print()
print("Sample: states for PR-00000:")
print(req_states[req_states['RequisitionID']=='PR-00000'])
print()
print("Sample: states for a rejected PR:")
pr_rej = req_states[req_states['State']=='Purchase Requisition Rejected']['RequisitionID'].iloc[0]
print(req_states[req_states['RequisitionID']==pr_rej])


State values:
State
Purchase Requisition Approved    2046
Purchase Order Confirmed         1830
Purchase Order Processed         1830
Purchase Requisition Rejected     454
Purchase Order Rejected           216
Name: count, dtype: int64

Sample: states for PR-00000:
    RequisitionID                          State  \
0        PR-00000  Purchase Requisition Approved   
125      PR-00000        Purchase Order Rejected   

                           Timestamp  
0    2026-04-22T08:51:43.050000+0000  
125  2026-05-02T14:31:42.578000+0000  

Sample: states for a rejected PR:
  RequisitionID                          State  \
5      PR-00007  Purchase Requisition Rejected   

                         Timestamp  
5  2026-04-23T08:40:33.807000+0000  


**Description:** This table records **system state-change events** for both PRs and POs. Each row represents a single instantaneous event — there is no duration (only one timestamp):
- `RequisitionID`: Refers to a PR ID (despite the name, it also records PO-level states).
- `State`: The state reached — one of: *Purchase Requisition Approved, Purchase Requisition Rejected, Purchase Order Confirmed, Purchase Order Rejected, Purchase Order Processed*.
- `Timestamp`: The moment the state was entered.

Since these are instantaneous events (no start/end), they will be modelled with `start_time = end_time`. The resource is the system (automated transition).


### 1.5 `logistics_table.csv` — Logistics Activities

In [10]:
print("Shape:", logistics.shape)
print()
logistics.head(6)


Shape: (7320, 5)



,purchase_order,employee,label,timestamp,cost
0,PO-0001,Operations Requester-000003,Receive Goods,2026-09-25T07:11:49.062000+0000,NaN
1,PO-0001,Operations Requester-000003,Goods Received,2026-09-25T08:47:46.173000+0000,95.95
2,PO-0001,Quality Specialist-000001,Inspect Goods,2026-09-25T10:15:15.819000+0000,NaN
3,PO-0001,Quality Specialist-000001,Goods Inspected,2026-09-25T10:26:43.981000+0000,13.38
4,PO-0002,Operations Requester-000001,Receive Goods,2026-06-09T12:19:53.011000+0000,NaN
5,PO-0002,Operations Requester-000001,Goods Received,2026-06-10T09:30:43.298000+0000,310.84


In [11]:
print("Label values:")
print(logistics['label'].value_counts())
print()
print("Sample: PO-0001 logistics events (sorted by time):")
print(logistics[logistics['purchase_order']=='PO-0001'].sort_values('timestamp').to_string())


Label values:
label
Receive Goods      1830
Goods Received     1830
Inspect Goods      1830
Goods Inspected    1830
Name: count, dtype: int64

Sample: PO-0001 logistics events (sorted by time):
  purchase_order                     employee            label                        timestamp   cost
0        PO-0001  Operations Requester-000003    Receive Goods  2026-09-25T07:11:49.062000+0000    NaN
1        PO-0001  Operations Requester-000003   Goods Received  2026-09-25T08:47:46.173000+0000  95.95
2        PO-0001    Quality Specialist-000001    Inspect Goods  2026-09-25T10:15:15.819000+0000    NaN
3        PO-0001    Quality Specialist-000001  Goods Inspected  2026-09-25T10:26:43.981000+0000  13.38


**Description:** This table records **warehouse/logistics activities** at the PO level. Crucially, it uses a **start-event / end-event** pattern:
- `Receive Goods` (no cost, no end) → `Goods Received` (has cost) form a matched pair for the *Receive Goods* activity.
- `Inspect Goods` (no cost) → `Goods Inspected` (has cost) form a matched pair for the *Inspect Goods* activity.

The `cost` column is `NaN` for start events and populated for end events. To convert to activity-instance format, start and end events must be matched and merged by `purchase_order` + `employee` + activity type.

Since data is at PO level, a join with `pr_po_table` is needed to obtain the PR case identifier.


### 1.6 `finance_table.csv` — Finance Activities

In [12]:
print("Shape:", finance.shape)
print()
finance.head(5)


Shape: (7320, 7)



,purchase_order,employee,label,cost,requisition_value,start,end
0,PO-0001,Financial Manager-000002,Receive Invoice,34.14,1213.82,2026-09-25T07:21:15.730000+0000,2026-09-25T07:50:31.541000+0000
1,PO-0001,Financial Manager-000002,Prepare Payment,5.29,1213.82,2026-09-25T07:51:37.642000+0000,2026-09-25T07:56:09.600000+0000
2,PO-0001,Financial Manager-000001,Authorize Payment,1.91,1213.82,2026-09-25T11:02:46.829000+0000,2026-09-25T11:04:25.244000+0000
3,PO-0001,Financial Manager-000001,Pay Invoice,35.75,1213.82,2026-09-25T11:09:16.300000+0000,2026-09-25T11:39:54.654000+0000
4,PO-0002,Financial Manager-000003,Receive Invoice,19.22,1325.97,2026-06-08T07:47:22.227000+0000,2026-06-08T08:03:50.529000+0000


In [13]:
print("Activity labels:")
print(finance['label'].value_counts())
print()
print("Sample: PO-0001 finance trace:")
print(finance[finance['purchase_order']=='PO-0001'].sort_values('start')[['purchase_order','label','employee','start','end']].to_string())


Activity labels:
label
Receive Invoice      1830
Prepare Payment      1830
Authorize Payment    1830
Pay Invoice          1830
Name: count, dtype: int64

Sample: PO-0001 finance trace:
  purchase_order              label                  employee                            start                              end
0        PO-0001    Receive Invoice  Financial Manager-000002  2026-09-25T07:21:15.730000+0000  2026-09-25T07:50:31.541000+0000
1        PO-0001    Prepare Payment  Financial Manager-000002  2026-09-25T07:51:37.642000+0000  2026-09-25T07:56:09.600000+0000
2        PO-0001  Authorize Payment  Financial Manager-000001  2026-09-25T11:02:46.829000+0000  2026-09-25T11:04:25.244000+0000
3        PO-0001        Pay Invoice  Financial Manager-000001  2026-09-25T11:09:16.300000+0000  2026-09-25T11:39:54.654000+0000


**Description:** This table records **finance department activities** at the PO level. Each row is already an activity instance with explicit start and end timestamps:
- `purchase_order`: PO identifier — must be joined via `pr_po_table` to get the PR case ID.
- `employee`: The financial officer or manager performing the task.
- `label`: Activity — one of: *Receive Invoice, Prepare Payment, Authorize Payment, Pay Invoice*.
- `start` / `end`: Activity start and end time.
- `cost`: Processing cost.
- `requisition_value`: Value of the associated requisition.


---
## Task 2: UML Class Diagram & Relationship Verification

### Schema Overview

The six tables form the following schema:

```
pr_info_table          pr_po_table        purchasing_table
─────────────          ───────────        ────────────────
PRID (PK)    ◄──────── PR (FK)            purchase_requisition (FK→pr_info.PRID)
...                    PO (FK) ────┐      employee, label
                                   │      timestamp_1, timestamp_2, cost
                                   │
                       ┌───────────┘
                       ▼
                  finance_table          logistics_table
                  ─────────────          ───────────────
                  purchase_order (FK)    purchase_order (FK)
                  employee, label        employee, label
                  start, end, cost       timestamp, cost

requisition_states_table
────────────────────────
RequisitionID (FK→pr_info.PRID)
State, Timestamp
```

### Relationship Verification


In [14]:
# Verify: pr_po.PR references pr_info.PRID
pr_ids = set(pr_info['PRID'])
pr_po_prs = set(pr_po['PR'])
print("All PR values in pr_po exist in pr_info:", pr_po_prs.issubset(pr_ids))
print("PR IDs in pr_po not in pr_info:", pr_po_prs - pr_ids)

# Verify: purchasing.purchase_requisition references pr_info.PRID
purch_prs = set(purchasing['purchase_requisition'])
print("\nAll PR values in purchasing exist in pr_info:", purch_prs.issubset(pr_ids))

# Verify: req_states.RequisitionID references pr_info.PRID
rs_ids = set(req_states['RequisitionID'])
print("All IDs in requisition_states exist in pr_info:", rs_ids.issubset(pr_ids))

# Verify: finance/logistics reference valid POs from pr_po
po_ids = set(pr_po['PO'])
finance_pos = set(finance['purchase_order'])
logistics_pos = set(logistics['purchase_order'])
print("\nAll POs in finance exist in pr_po:", finance_pos.issubset(po_ids))
print("All POs in logistics exist in pr_po:", logistics_pos.issubset(po_ids))


All PR values in pr_po exist in pr_info: True
PR IDs in pr_po not in pr_info: set()

All PR values in purchasing exist in pr_info: True
All IDs in requisition_states exist in pr_info: True

All POs in finance exist in pr_po: True
All POs in logistics exist in pr_po: True


**UML Class Diagram** (described textually — see accompanying diagram image):

- `PR_Info` **1 ──── 0..1** `PR_PO` (a PR may or may not result in a PO)
- `PR_PO` **1 ──── 0..*` `Finance_Table` (one PO → multiple finance activities)
- `PR_PO` **1 ──── 0..*` `Logistics_Table` (one PO → multiple logistics events)
- `PR_Info` **1 ──── 0..*` `Purchasing_Table` (one PR → multiple purchasing activities)
- `PR_Info` **1 ──── 0..*` `Requisition_States_Table` (one PR → zero or more state events)


---
## Task 3: Preprocessing Tables into Activity Instance Format

Each table must be transformed so that every row represents one activity instance with: `case_id`, `activity`, `resource`, `start_time`, `end_time`.


### 3.1 Preprocessing `purchasing_table`

In [15]:
# Already in activity-instance format: rename columns
purchasing_clean = purchasing.rename(columns={
    'purchase_requisition': 'case_id',
    'employee': 'resource',
    'label': 'activity',
    'timestamp_1': 'start_time',
    'timestamp_2': 'end_time'
})[['case_id','activity','resource','start_time','end_time','cost']].copy()

print("purchasing_clean shape:", purchasing_clean.shape)
print(purchasing_clean.head(3).to_string())


purchasing_clean shape: (10394, 6)
    case_id                      activity                     resource                       start_time                         end_time    cost
0  PR-00000   Create Purchase Requisition  Operations Requester-000001  2026-04-22T07:00:00.000000+0000  2026-04-22T08:09:35.186000+0000   69.59
1  PR-00001   Create Purchase Requisition  Operations Requester-000002  2026-04-22T07:05:28.919000+0000  2026-04-22T09:43:47.141000+0000  158.30
2  PR-00000  Analyze Purchase Requisition    Purchasing Officer-000001  2026-04-22T08:09:35.186000+0000  2026-04-22T08:51:42.050000+0000   35.10


### 3.2 Preprocessing `logistics_table` (start/end event pairing)

In [16]:
# The table stores paired events:
#   start event: 'Receive Goods' / 'Inspect Goods'  (cost = NaN)
#   end event:   'Goods Received' / 'Goods Inspected' (cost = filled)
# We need to join them on (purchase_order, employee, activity)

end_label_map = {'Goods Received': 'Receive Goods', 'Goods Inspected': 'Inspect Goods'}

log_start = logistics[logistics['label'].isin(['Receive Goods','Inspect Goods'])].copy()
log_start = log_start.rename(columns={'timestamp':'start_time','label':'activity'})

log_end = logistics[logistics['label'].isin(end_label_map.keys())].copy()
log_end['activity'] = log_end['label'].map(end_label_map)
log_end = log_end.rename(columns={'timestamp':'end_time'})[['purchase_order','employee','activity','end_time','cost']]

logistics_paired = log_start[['purchase_order','employee','activity','start_time']].merge(
    log_end, on=['purchase_order','employee','activity'], how='inner'
)

print("Paired rows:", len(logistics_paired))
print("Unpaired start events:", len(log_start) - len(logistics_paired))
print(logistics_paired.head(3).to_string())


Paired rows: 3660
Unpaired start events: 0
  purchase_order                     employee       activity                       start_time                         end_time    cost
0        PO-0001  Operations Requester-000003  Receive Goods  2026-09-25T07:11:49.062000+0000  2026-09-25T08:47:46.173000+0000   95.95
1        PO-0001    Quality Specialist-000001  Inspect Goods  2026-09-25T10:15:15.819000+0000  2026-09-25T10:26:43.981000+0000   13.38
2        PO-0002  Operations Requester-000001  Receive Goods  2026-06-09T12:19:53.011000+0000  2026-06-10T09:30:43.298000+0000  310.84


In [17]:
# Join PO → PR
po_pr_map = pr_po.rename(columns={'PR':'case_id','PO':'purchase_order'})
logistics_clean = logistics_paired.merge(po_pr_map, on='purchase_order', how='left')
logistics_clean = logistics_clean.rename(columns={'employee':'resource'})
logistics_clean = logistics_clean[['case_id','activity','resource','start_time','end_time','cost']].copy()

print("logistics_clean shape:", logistics_clean.shape)
print(logistics_clean.head(3).to_string())


logistics_clean shape:

 (3660, 6)
    case_id       activity                     resource                       start_time                         end_time    cost
0  PR-01179  Receive Goods  Operations Requester-000003  2026-09-25T07:11:49.062000+0000  2026-09-25T08:47:46.173000+0000   95.95
1  PR-01179  Inspect Goods    Quality Specialist-000001  2026-09-25T10:15:15.819000+0000  2026-09-25T10:26:43.981000+0000   13.38
2  PR-00335  Receive Goods  Operations Requester-000001  2026-06-09T12:19:53.011000+0000  2026-06-10T09:30:43.298000+0000  310.84


### 3.3 Preprocessing `finance_table`

In [18]:
# Already activity-instance format, but PO-level: need PO→PR join
finance_clean = finance.rename(columns={
    'purchase_order':'po_id','employee':'resource','label':'activity',
    'start':'start_time','end':'end_time'
})[['po_id','activity','resource','start_time','end_time','cost']].copy()

po_pr_map2 = pr_po.rename(columns={'PR':'case_id','PO':'po_id'})
finance_clean = finance_clean.merge(po_pr_map2, on='po_id', how='left')
finance_clean = finance_clean[['case_id','activity','resource','start_time','end_time','cost']].copy()

print("finance_clean shape:", finance_clean.shape)
print(finance_clean.head(3).to_string())


finance_clean shape: (7320, 6)
    case_id           activity                  resource                       start_time                         end_time   cost
0  PR-01179    Receive Invoice  Financial Manager-000002  2026-09-25T07:21:15.730000+0000  2026-09-25T07:50:31.541000+0000  34.14
1  PR-01179    Prepare Payment  Financial Manager-000002  2026-09-25T07:51:37.642000+0000  2026-09-25T07:56:09.600000+0000   5.29
2  PR-01179  Authorize Payment  Financial Manager-000001  2026-09-25T11:02:46.829000+0000  2026-09-25T11:04:25.244000+0000   1.91


### 3.4 Preprocessing `requisition_states_table` (instantaneous events)

In [19]:
# State transitions are instantaneous: set start_time = end_time
req_clean = req_states.rename(columns={
    'RequisitionID':'case_id','State':'activity','Timestamp':'start_time'
}).copy()
req_clean['end_time'] = req_clean['start_time']
req_clean['resource'] = 'System'
req_clean['cost'] = None
req_clean = req_clean[['case_id','activity','resource','start_time','end_time','cost']].copy()

print("req_clean shape:", req_clean.shape)
print(req_clean.head(3).to_string())


req_clean shape: (6376, 6)
    case_id                       activity resource                       start_time                         end_time  cost
0  PR-00000  Purchase Requisition Approved   System  2026-04-22T08:51:43.050000+0000  2026-04-22T08:51:43.050000+0000  None
1  PR-00002  Purchase Requisition Approved   System  2026-04-22T10:35:08.574000+0000  2026-04-22T10:35:08.574000+0000  None
2  PR-00003  Purchase Requisition Approved   System  2026-04-22T10:37:55.319000+0000  2026-04-22T10:37:55.319000+0000  None


---
## Task 4: Merging into the Final Activity Instance Log


In [20]:
# Concatenate all preprocessed tables
event_log = pd.concat([
    purchasing_clean.assign(source='purchasing'),
    logistics_clean.assign(source='logistics'),
    finance_clean.assign(source='finance'),
    req_clean.assign(source='requisition_states'),
], ignore_index=True)

# Add case attributes from pr_info
event_log = event_log.merge(
    pr_info.rename(columns={'PRID':'case_id'}),
    on='case_id', how='left'
)

# Parse timestamps and sort
event_log['start_time'] = pd.to_datetime(event_log['start_time'], utc=True)
event_log['end_time']   = pd.to_datetime(event_log['end_time'], utc=True)
event_log = event_log.sort_values(['case_id','start_time']).reset_index(drop=True)

print("Final event log shape:", event_log.shape)
print("Total cases:", event_log['case_id'].nunique())
print("Activities:")
print(event_log['activity'].value_counts())


Final event log shape:

 (27750, 12)
Total cases: 2500
Activities:
activity
Analyze Purchase Requisition     3151
Create Purchase Requisition      2500
Purchase Requisition Approved    2046
Create Purchase Order            2046
Confirm PO with Vendor           2046
Purchase Order Confirmed         1830
Receive Goods                    1830
Inspect Goods                    1830
Receive Invoice                  1830
Prepare Payment                  1830
Authorize Payment                1830
Pay Invoice                      1830
Purchase Order Processed         1830
Amend Purchase Requisition        651
Purchase Requisition Rejected     454
Purchase Order Rejected           216
Name: count, dtype: int64


In [21]:
# Preview a full trace
print("Full trace for PR-00000:")
print(event_log[event_log['case_id']=='PR-00000'][
    ['case_id','activity','resource','start_time','end_time']
].to_string())


Full trace for PR-00000:
    case_id                       activity                     resource                       start_time                         end_time
0  PR-00000    Create Purchase Requisition  Operations Requester-000001        2026-04-22 07:00:00+00:00 2026-04-22 08:09:35.186000+00:00
1  PR-00000   Analyze Purchase Requisition    Purchasing Officer-000001 2026-04-22 08:09:35.186000+00:00 2026-04-22 08:51:42.050000+00:00
2  PR-00000  Purchase Requisition Approved                       System 2026-04-22 08:51:43.050000+00:00 2026-04-22 08:51:43.050000+00:00
3  PR-00000          Create Purchase Order    Purchasing Officer-000002 2026-04-22 08:51:44.050000+00:00 2026-04-22 08:53:21.313000+00:00
4  PR-00000         Confirm PO with Vendor    Purchasing Officer-000003 2026-04-22 08:53:21.313000+00:00 2026-05-02 14:31:41.578000+00:00
5  PR-00000        Purchase Order Rejected                       System 2026-05-02 14:31:42.578000+00:00 2026-05-02 14:31:42.578000+00:00


In [22]:
# Summary statistics
print("Events per case:")
print(event_log.groupby('case_id').size().describe())
print()
print("Null case_ids (failed PO→PR join):", event_log['case_id'].isna().sum())
print()
print("Date range:")
print("  From:", event_log['start_time'].min())
print("  To:  ", event_log['end_time'].max())


Events per case:
count    2500.000000
mean       11.100000
std         4.059925
min         3.000000
25%         8.000000
50%        13.000000
75%        13.000000
max        15.000000
dtype: float64

Null case_ids (failed PO→PR join): 0

Date range:
  From: 2026-04-22 07:00:00+00:00
  To:   2027-03-12 07:42:54.831000+00:00


---
## Task 5: Export for Apromore


In [23]:
# Export to CSV
event_log.to_csv('event_log_p2p.csv', index=False)
print("Saved: event_log_p2p.csv")
print("Rows:", len(event_log))
print("Columns:", event_log.columns.tolist())


Saved: event_log_p2p.csv
Rows: 27750
Columns: ['case_id', 'activity', 'resource', 'start_time', 'end_time', 'cost', 'source', 'InitialRequisitionValue', 'PurchasingCategory', 'VendorID', 'Urgency', 'CostCenter']


### Loading into Apromore — Step-by-Step

1. Go to [https://academic-cloud.apromore.org/](https://academic-cloud.apromore.org/) and log in.
2. Click **Upload** (top toolbar) → select `event_log_p2p.csv`.
3. In the import dialog, map columns:
   - **Case ID** → `case_id`
   - **Activity** → `activity`
   - **Start Timestamp** → `start_time`
   - **End Timestamp** → `end_time`
   - **Resource** → `resource`
4. Click **OK** to import.
5. Right-click the log → **Discover** → **Discover Process Model (BPMN)**.
6. Set parallelism slider to **Full** (100%).
7. Take a screenshot of the resulting BPMN diagram.

### Process Control-Flow Analysis

The P2P process follows this structure:
- **Start**: Every case begins with *Create Purchase Requisition*.
- **Sequential block (Purchasing)**: Analyze → (optionally Amend → re-Analyze loop) → system state event.
- **Gateway**: *Purchase Requisition Approved* → continues; *Purchase Requisition Rejected* → case ends.
- **After approval**: *Create Purchase Order* → *Confirm PO with Vendor* → system state.
- **Another gateway**: *Purchase Order Confirmed* → parallel split into logistics + finance; *Purchase Order Rejected* → case ends.
- **Parallel branch 1 (Logistics)**: *Receive Goods* then *Inspect Goods* (sequential).
- **Parallel branch 2 (Finance)**: *Receive Invoice* → *Prepare Payment* → *Authorize Payment* → *Pay Invoice* (sequential).
- **Join**: Both parallel branches converge at *Purchase Order Processed* (system event) → case ends.

See screenshot in Deliverable 3.
